# Judge the split-trained 0.5B DPO candidate bank with GPT

This notebook annotates the 2,260 candidates generated for the 226 training questions held out from SFT. Exact accepted-alias matches are assigned C3 locally. Non-exact candidates are sent to the configured GPT judge with the question, accepted aliases, and supplied PubMed snippets for C2/C1/UNCERTAIN classification.

The workflow is cached and resumable. It then injects an extractive gold C3 anchor, builds all ordered class pairs, and writes the three curriculum stages with JSONL and CSV audit files. Running the annotation cell sends the stated records to the OpenAI API.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'cse_dpo/annotate_split_dpo_candidate_bank.py').is_file():
            return candidate
    raise FileNotFoundError('Could not find the project root')

PROJECT_ROOT = find_project_root(Path.cwd())
BANK = PROJECT_ROOT / 'Artifacts/cse_dpo/candidate_banks/qwen25_05b_sft80_dpo20_r32_step175_dpo226_s10_t07_seed3407/adapter-best-evidence-mrr/candidate_bank.jsonl'
QUESTIONS = PROJECT_ROOT / 'data/BioASQ_factoid_sft_prepared/gold_supported_question_split_sft80_dpo20_seed3407/dpo_questions_for_candidate_generation.json'
API_KEY_FILE = PROJECT_ROOT / 'open_ai_api.txt'
OUTPUT_ROOT = PROJECT_ROOT / 'Artifacts/cse_dpo/class_judgments/qwen25_05b_sft80_dpo20_r32_step175_dpo226_s10_t07_seed3407_gold_c3'

JUDGE_MODEL = os.environ.get('CANDIDATE_BANK_JUDGE_MODEL', 'gpt-4.1-mini-2025-04-14')
MAX_NEW_JUDGE_CALLS = 500
REQUEST_DELAY_SECONDS = 1.0
RUN_ANNOTATION = True

for path in (BANK, QUESTIONS, API_KEY_FILE):
    assert path.exists(), f'Missing required input: {path}'
print(json.dumps({
    'bank': str(BANK),
    'questions': str(QUESTIONS),
    'judge_model': JUDGE_MODEL,
    'output_root': str(OUTPUT_ROOT),
    'run_annotation': RUN_ANNOTATION,
}, indent=2))

## Inspect the annotation plan

This cell performs local validation only. It makes no API calls.

In [ ]:
dry_run_command = [
    sys.executable, '-m', 'cse_dpo.annotate_split_dpo_candidate_bank',
    '--bank', str(BANK),
    '--questions', str(QUESTIONS),
    '--output-root', str(OUTPUT_ROOT),
    '--api-key-file', str(API_KEY_FILE),
    '--judge-model', JUDGE_MODEL,
    '--max-new-judge-calls', str(MAX_NEW_JUDGE_CALLS),
    '--request-delay-seconds', str(REQUEST_DELAY_SECONDS),
    '--dry-run',
]
subprocess.run(dry_run_command, cwd=PROJECT_ROOT, check=True)
plan = json.loads((OUTPUT_ROOT / 'annotation_plan.json').read_text(encoding='utf-8'))
print(json.dumps(plan['selection'], indent=2))

## Run GPT annotation and construct pairs

This cell sends non-exact records to the OpenAI API. Exact C3 records remain local. Successful responses are cached individually, so rerunning after an interruption continues without paying for completed judgments again.

In [ ]:
run_command = [item for item in dry_run_command if item != '--dry-run']
print(' '.join(run_command[:-8] + ['<API settings omitted from display>']))
if RUN_ANNOTATION:
    subprocess.run(run_command, cwd=PROJECT_ROOT, check=True)
else:
    print('Annotation disabled. Set RUN_ANNOTATION=True in the configuration cell.')

## Review results

In [ ]:
MANIFEST = OUTPUT_ROOT / 'manifest.json'
assert MANIFEST.exists(), f'Annotation is not complete: {MANIFEST}'
manifest = json.loads(MANIFEST.read_text(encoding='utf-8'))
judgment_summary = manifest['judgment_summary']
pair_summary = manifest['pair_summary']
stage_summary = manifest['stage_summary']['curriculum']
print(json.dumps({
    'status': manifest['status'],
    'class_counts': judgment_summary['class_counts'],
    'new_api_calls': judgment_summary['new_api_calls'],
    'rate_limit_hits': judgment_summary['rate_limit_hits'],
    'pair_count': pair_summary['pair_count'],
    'questions_with_pairs': pair_summary['questions_with_pairs'],
    'stage_counts': {stage: values['pair_count'] for stage, values in stage_summary.items()},
}, indent=2))
print('Judgment CSV:', OUTPUT_ROOT / 'candidate_class_judgments.csv')
print('Pair CSV:', OUTPUT_ROOT / 'dpo_pairs.csv')
print('Curriculum directory:', OUTPUT_ROOT / 'staged_curriculum_pairs')

In [ ]:
judgments = [json.loads(line) for line in (OUTPUT_ROOT / 'candidate_class_judgments.jsonl').open(encoding='utf-8') if line.strip()]
nonexact_preview = [row for row in judgments if not row['exact_match']][:10]
for row in nonexact_preview:
    print({
        'question_id': row['question_id'],
        'candidate': row['candidate'],
        'class': row['class'],
        'evidence_support': row['evidence_support'],
        'confidence': row['confidence'],
        'basis': row['basis'],
    })